In [6]:
experiment = "vggnet16_1_ten_imgs"

In [2]:
# To sort the results.csv by instance_id
CSV_PATH = f"../results/{experiment}/results.csv"

import pandas as pd
df = pd.read_csv(CSV_PATH)
df["instance_id"] = pd.to_numeric(df["instance_id"], errors="coerce")
df_sorted = df.sort_values(by="instance_id", ascending=True)
df_sorted.to_csv(CSV_PATH, index=False)

In [28]:
import pandas as pd
import re
from pathlib import Path

results_path = f"../results/{experiment}/results.csv"
stats_path = f"../results/{experiment}/input_change_stats.csv"
out_path = f"../results/{experiment}/combined_results.csv"

df_res = pd.read_csv(results_path)
df_ics = pd.read_csv(stats_path)

print("results.csv:", df_res.shape)
print("input_change_stats.csv:", df_ics.shape)

from pathlib import Path

TAG_MAP = {
    "global": "global",
    "fixmask": "fix_mask",
    "fixnonmask": "fix_nonmask",
    "fix_mask": "fix_mask",
    "fix_nonmask": "fix_nonmask",
}

def normalize_eps_str(eps_str: str) -> str:
    """Normalize numeric strings like '0.00010' -> '0.0001', '1e-4' -> '0.0001' (best-effort)."""
    s = str(eps_str).strip()
    # if already plain decimal
    if re.fullmatch(r"[0-9]*\.?[0-9]+", s):
        # strip trailing zeros after decimal
        if "." in s:
            s = s.rstrip("0").rstrip(".")
        return s
    # scientific notation fallback
    try:
        v = float(s)
        # use a stable non-scientific representation
        s2 = f"{v:.20f}".rstrip("0").rstrip(".")
        return s2
    except Exception:
        return s

def parse_vnnlib_fields(vnnlib_path: str):
    """
    Examples:
      vnnlib/n01443537_goldfish_global_k50176_eps_0.0001.vnnlib
      vnnlib/n01443537_goldfish_seg0_fixmask_k3136_eps_0.0001.vnnlib
    Returns: image, tag, segment_index, k, eps_key
    """
    base = Path(vnnlib_path).name
    stem = base.replace(".vnnlib", "")

    # global
    m = re.match(r"^(?P<image>.+?)_global_k(?P<k>\d+)_eps_(?P<eps>[^_]+)$", stem)
    if m:
        return {
            "image": m.group("image"),
            "tag": "global",
            "segment_index": -1,
            "k": int(m.group("k")),
            "eps_key": normalize_eps_str(m.group("eps")),
        }

    # segX_fixmask / segX_fixnonmask
    m = re.match(
        r"^(?P<image>.+?)_seg(?P<seg>\d+)_(?P<tag>fixmask|fixnonmask|fix_mask|fix_nonmask)_k(?P<k>\d+)_eps_(?P<eps>[^_]+)$",
        stem,
    )
    if m:
        return {
            "image": m.group("image"),
            "tag": TAG_MAP.get(m.group("tag"), m.group("tag")),
            "segment_index": int(m.group("seg")),
            "k": int(m.group("k")),
            "eps_key": normalize_eps_str(m.group("eps")),
        }

    return {"image": None, "tag": None, "segment_index": None, "k": None, "eps_key": None}

def parse_model_from_onnx(onnx_path: str):
    # onnx/vgg16-7.onnx  -> vgg16-7
    base = Path(onnx_path).name
    return base.replace(".onnx", "")

parsed = df_res["vnnlib"].apply(parse_vnnlib_fields).apply(pd.Series)
df_res2 = pd.concat([df_res, parsed], axis=1)
df_res2["model"] = df_res2["onnx"].apply(parse_model_from_onnx)

# Normalize tags if results already had them (safety)
df_res2["tag"] = df_res2["tag"].map(lambda x: TAG_MAP.get(x, x))

print("Failed vnnlib parses:", df_res2["image"].isna().sum())

# Build eps_key in input_change_stats and normalize tag there too
df_ics2 = df_ics.copy()

df_ics2["model"] = df_ics2["model"].astype(str)
df_ics2["image"] = df_ics2["image"].astype(str)
df_ics2["tag"] = df_ics2["tag"].astype(str).map(lambda x: TAG_MAP.get(x, x))
df_ics2["segment_index"] = df_ics2["segment_index"].astype(int)
df_ics2["k"] = df_ics2["k"].astype(int)
df_ics2["eps_key"] = df_ics2["eps"].map(normalize_eps_str)

# Make sure results types match
df_res2["model"] = df_res2["model"].astype(str)
df_res2["image"] = df_res2["image"].astype(str)
df_res2["segment_index"] = df_res2["segment_index"].astype(int)
df_res2["k"] = df_res2["k"].astype(int)
df_res2["eps_key"] = df_res2["eps_key"].astype(str)

KEYS = ["image", "model", "tag", "segment_index", "k", "eps_key"]

merged = df_res2.merge(df_ics2, on=KEYS, how="left", suffixes=("", "_ics"))

print("Merged shape:", merged.shape)

# Check matches using a column that should exist in input_change_stats:
probe_col = "num_changed" if "num_changed" in merged.columns else None
if probe_col:
    no_match = merged[probe_col].isna().sum()
    print("Rows with no match in input_change_stats:", no_match, "out of", len(merged))
else:
    print("Could not find 'num_changed' column to probe matches; check your input_change_stats columns.")

# Show the specific example row(s) for goldfish global k50176 eps 0.0001
example = merged[
    (merged["image"] == "n01443537_goldfish")
    & (merged["model"] == "vgg16-7")
    & (merged["tag"] == "global")
    & (merged["segment_index"] == -1)
    & (merged["k"] == 50176)
    & (merged["eps_key"] == "0.0001")
]
# If there are still unmatched rows, inspect a few
if "num_changed" in merged.columns:
    unmatched = merged[merged["num_changed"].isna()][["vnnlib", "onnx"] + KEYS].copy()
    print("Unmatched examples (first 30):")
    display(unmatched.head(30))

merged.to_csv(out_path, index=False)
print("Wrote:", out_path)


results.csv: (144, 10)
input_change_stats.csv: (144, 14)
Failed vnnlib parses: 0
Merged shape: (144, 25)
Rows with no match in input_change_stats: 0 out of 144
Unmatched examples (first 30):


,vnnlib,onnx,image,model,tag,segment_index,k,eps_key


Wrote: ../results/vggnet16_1_ten_imgs/combined_results.csv


In [ ]:
# To Combining the results of two csv files
import pandas as pd

df1 = pd.read_csv("../results/vggnet16_benchmark2022_segmented_all/results.csv")
df2 = pd.read_csv("../results/vggnet16_benchmark2022_segmented_all/results_part1_until_5700.csv")

combined = pd.concat([df1, df2], ignore_index=True)

combined.to_csv("../results/vggnet16_benchmark2022_segmented_all/results_part1_until_5700.csv", index=False)

In [24]:
# To filter a dataframe to only images that appear exactly 3 times (G O B)
import pandas as pd
import numpy as np

def complete_triplets(df, image_col="image", debug_path=None):
    df = df.copy()
    print("Original shape:", df.shape)

    # keep only images that appear exactly 3 times
    counts = df[image_col].value_counts()
    good_images = counts[counts == 3].index
    df = df[df[image_col].isin(good_images)].copy()

    print("Number of images:", len(good_images))
    print("Shape after filtering to triplets:", df.shape)

    if debug_path is not None:
        df.to_csv(f"{debug_path}_{len(good_images)}_imgs.csv", index=False)


CSV_PATH = f"../../results/{experiment}/combined_results.csv"
OUT_DIR  = f"../results/{experiment}"

df = pd.read_csv(CSV_PATH)

# numeric safety
df["k"] = pd.to_numeric(df["k"], errors="coerce")
df["eps"] = pd.to_numeric(df["eps"], errors="coerce")

# (k, eps) -> subdf
dfs_by_keps = {key: subdf.copy() for key, subdf in df.groupby(["k", "eps"])}

# filter and save each (k, eps)
dfs_by_keps_triplets = {}
for (k, eps), subdf in dfs_by_keps.items():
    print(f"Processing k={k}, eps={eps}")
    dbg = f"{OUT_DIR}/triplets_k{k}_eps{eps}"
    complete_triplets(subdf, debug_path=dbg)

FileNotFoundError: [Errno 2] No such file or directory: '../../results/vggnet16_1_ten_imgs/combined_results.csv'

In [ ]:
# To find BnB Instance (`domains_visited > 0`)
import os
import pandas as pd

experiment = "exp_1/triplets_k50176.0_eps0.0001_239_imgs"

CSV_PATH = f"results/{experiment}.csv"

df = pd.read_csv(CSV_PATH)

assert "domains_visited" in df.columns, "domains_visited column not found!"

df["domains_visited"] = pd.to_numeric(df["domains_visited"], errors="coerce").fillna(0)

bnb_df = df[df["domains_visited"] > 0].copy()

bnb_df = bnb_df.sort_values(
    by="lb_minus_rhs",
    ascending=False
)

print("Rows with domains_visited > 0:", len(bnb_df))
bnb_df.head()


cols = [
    "instance_id",
    "image",
    "tag",
    "is_global",
    "segment_index",
    "eps",
    "k",
    "result",
    "lb_minus_rhs",
    "domains_visited",
    "bab_time",
    "all_time",
]

# Only keep columns that exist (safe if your CSV is slightly different)
cols = [c for c in cols if c in bnb_df.columns]

bnb_df[cols].sort_values("domains_visited", ascending=False)

# OUT_PATH = f"rows_with_bnb_entered_{experiment}.csv"
# os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
# bnb_df.to_csv(OUT_PATH, index=False)
# print("Saved to", OUT_PATH)

In [3]:
# To show the table
import csv

CSV = "csvs/all_vggnet16_results_k_all.csv"   # your merged file

with open(CSV, newline="") as f:
    r = csv.reader(f)
    rows = list(r)

# print as a simple aligned table
widths = [max(len(str(x)) for x in col) for col in zip(*rows)]
for i, row in enumerate(rows):
    line = " | ".join(str(x).ljust(w) for x, w in zip(row, widths))
    print(line)
    if i == 0:
        print("-+-".join("-"*w for w in widths))

vnnlib                                                     | timeout | result        | all_time           | onnx              | instance_id | lb_minus_rhs        | bab_time           | domains_visited | init_unstable | source_folder                               
-----------------------------------------------------------+---------+---------------+--------------------+-------------------+-------------+---------------------+--------------------+-----------------+---------------+---------------------------------------------
vnnlib/n02033041_dowitcher_global_k50176_eps_0.0001.vnnlib | 1200    | timeout False | 4005.6531410217285 | onnx/vgg16-7.onnx | 1           | -263.40545654296875 | 3988.4435873031616 | 3               | 0             | vggnet16_benchmark2022_one_img_naive        
vnnlib/n02033041_dowitcher_global_k50176_eps_0.0001.vnnlib | 1200    | timeout False | 4001.934016942978  | onnx/vgg16-7.onnx | 1           | -263.4098205566406  | 3984.5712988376617 | 3               | 0    

In [6]:
# To show the table
import csv

CSV = "csvs/all_vggnet16_results_k_500.csv"   # your merged file

with open(CSV, newline="") as f:
    r = csv.reader(f)
    rows = list(r)

# print as a simple aligned table
widths = [max(len(str(x)) for x in col) for col in zip(*rows)]
for i, row in enumerate(rows):
    line = " | ".join(str(x).ljust(w) for x, w in zip(row, widths))
    print(line)
    if i == 0:
        print("-+-".join("-"*w for w in widths))

vnnlib                                                   | timeout | result      | all_time           | onnx              | instance_id | lb_minus_rhs       | bab_time           | domains_visited | init_unstable | source_folder                                    
---------------------------------------------------------+---------+-------------+--------------------+-------------------+-------------+--------------------+--------------------+-----------------+---------------+--------------------------------------------------
vnnlib/n02033041_dowitcher_global_k500_eps_0.0001.vnnlib | 1200    | unsat False | 842.1632499694824  | onnx/vgg16-7.onnx | 1           | 1.3206205368041992 | 794.1718530654907  | 0               | 0             | vggnet16_benchmark2022_one_img_naive_k500        
vnnlib/n02033041_dowitcher_global_k500_eps_0.0001.vnnlib | 1200    | unsat False | 804.6202256679535  | onnx/vgg16-7.onnx | 1           | 1.3206206560134888 | 793.8658721446991  | 0               | 0         

In [5]:
# To show the table
import csv

CSV = "csvs/all_vggnet16_results_ks_12_14.csv"   # your merged file

with open(CSV, newline="") as f:
    r = csv.reader(f)
    rows = list(r)

# print as a simple aligned table
widths = [max(len(str(x)) for x in col) for col in zip(*rows)]
for i, row in enumerate(rows):
    line = " | ".join(str(x).ljust(w) for x, w in zip(row, widths))
    print(line)
    if i == 0:
        print("-+-".join("-"*w for w in widths))

vnnlib                                                              | domains_visited | init_unstable | result        | timeout | instance_id | lb_minus_rhs        | bab_time           | all_time           | onnx              | source_folder                                  
--------------------------------------------------------------------+-----------------+---------------+---------------+---------+-------------+---------------------+--------------------+--------------------+-------------------+------------------------------------------------
vnnlib/n02033041_dowitcher_global_k50176_eps_0.0001.vnnlib          |                 |               | error         | 1200    | 1           |                     |                    |                    | onnx/vgg16-7.onnx | vggnet16_benchmark2022_one_img_original        
vnnlib/n02033041_dowitcher_seg0_fixmask_k50176_eps_0.0001.vnnlib    | 0               | 0             | unsat False   | 1200    | 2           | 1.3113386631011963  | 885.98